# scm2cpp-lasso against scikit-learn

[scm2cpp](https://github.com/niitsuma/scm2cpp) translates scikit-learn's algorithms, written plainly in Scheme, into C++; [`scm2cpp-lasso`](https://pypi.org/project/scm2cpp-lasso/) is that C++ packaged for Python. This notebook installs it and times it against scikit-learn on the machine it runs on, checking that both give the same answer.

* **CPU runtime** (the default): installs the wheel, nothing to compile, a few minutes for everything below.
* **GPU runtime** (*Runtime -> Change runtime type -> GPU*): installs from the source distribution so that the CUDA builds are compiled for this GPU (a few more minutes), and the last section runs on the device.

The times are this machine's: Colab gives two virtual CPUs and, with a GPU runtime, a T4, so they are not the README's (one core of an i9-10900X, an RTX 4090). scikit-learn's BLAS is pinned to one thread, as the package's kernels are single-threaded.

In [ ]:
import subprocess, sys
gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if gpu:
    # the sdist builds the CUDA paths where nvcc is on the path
    !pip install -q --no-binary scm2cpp-lasso scm2cpp-lasso scikit-learn
else:
    !pip install -q scm2cpp-lasso scikit-learn

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
import time, warnings
import numpy as np, sklearn
import scm2cpp_lasso as m
warnings.simplefilter("ignore")
print("scm2cpp_lasso", m.__version__, "| scikit-learn", sklearn.__version__,
      "| device:", m.cuda_available(), "| derived lasso backend:", m.derived_backend())

def best(f, k=2):
    t, r = [], None
    for _ in range(k):
        t0 = time.perf_counter(); r = f(); t.append(time.perf_counter() - t0)
    return min(t), r

rows = []
def report(name, t_sk, t_us, diff, note=""):
    rows.append((name, t_sk, t_us, diff, note))
    print(f"{name:44} sklearn {t_sk:8.3f} s   scm2cpp {t_us:8.3f} s   {t_sk / t_us:6.1f}x   max diff {diff:.1e} {note}")

## Lasso: a grid of penalties, each from zero

The shape of a cross-validation grid, where warm starts are not available: every penalty solved from a zero start. scikit-learn fits `Lasso(fit_intercept=False)` once per penalty (the problem `CovLasso` solves); `CovLasso` forms the Gram matrix once and sweeps over it (`fit_path_batch`; on a GPU runtime it runs one block per penalty on the device). The translated side solves to `tol=1e-8`, scikit-learn to its default `1e-4`; the agreement is checked against a tight scikit-learn solve (`tol=1e-12`, untimed).

In [ ]:
from sklearn.linear_model import Lasso
rng = np.random.default_rng(0)
n, p, L = 1800, 200, 512
X = rng.standard_normal((n, p)); beta = np.zeros(p); beta[:10] = rng.standard_normal(10)
y = X @ beta + 0.5 * rng.standard_normal(n)
model = m.CovLasso(X, y)
lams = model.lambda_grid(num=L, eps=1e-2)
t_sk, sk = best(lambda: np.array([Lasso(alpha=a, fit_intercept=False).fit(X, y).coef_ for a in lams]), 1)
t_us, us = best(lambda: m.CovLasso(X, y).fit_path_batch(lams))
# agreement against a tight scikit-learn solve (untimed) at every 64th penalty
ref = np.array([Lasso(alpha=a, fit_intercept=False, tol=1e-12, max_iter=100000).fit(X, y).coef_ for a in lams[::64]])
report(f"Lasso, {L} penalties from zero, {n} x {p}", t_sk, t_us, np.abs(us[::64] - ref).max(), "(vs a tight solve)")

## LassoCV

Five folds, 100 penalties. `CovLassoCV` takes each fold's training Gram as the full Gram minus the fold's.

In [ ]:
from sklearn.linear_model import LassoCV
n, p = 20000, 200
X = rng.standard_normal((n, p)); y = X[:, :10] @ rng.standard_normal(10) + rng.standard_normal(n)
t_sk, sk = best(lambda: LassoCV(cv=5).fit(X, y), 1)
t_us, us = best(lambda: m.CovLassoCV(cv=5).fit(X, y))
report(f"LassoCV, {n} x {p}", t_sk, t_us, np.abs(us.coef_ - sk.coef_).max(), f"(alpha {sk.alpha_:.4g} / {us.alpha_:.4g})")

## Logistic regression (L-BFGS)

scipy's L-BFGS-B as scikit-learn runs it, translated: the same iterations, the same coefficients to rounding. On a GPU runtime the package takes its `--cuda` build for problems of a million entries or more.

In [ ]:
from sklearn.linear_model import LogisticRegression
n, p = 200000, 100
X = rng.standard_normal((n, p)) / 10
yb = (X @ rng.standard_normal(p) + rng.logistic(size=n) > 0).astype(int)
t_sk, sk = best(lambda: LogisticRegression().fit(X, yb), 1)
t_us, us = best(lambda: m.LBFGSLogisticRegression().fit(X, yb))
report(f"LogisticRegression, {n} x {p}", t_sk, t_us, np.abs(us.coef_ - sk.coef_).max(),
       f"({int(np.max(sk.n_iter_))} / {int(np.max(us.n_iter_))} iterations)")

## Recursive feature elimination around ridge

scikit-learn refits the ridge on every round; `GramRFE` forms the Gram once and takes each dropped feature out of its Cholesky factor.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge
n, p = 10000, 200
X = rng.standard_normal((n, p)); y = X[:, :10] @ rng.standard_normal(10) + rng.standard_normal(n)
t_sk, sk = best(lambda: RFE(Ridge(alpha=1.0), n_features_to_select=20).fit(X, y), 1)
t_us, us = best(lambda: m.GramRFE(1.0, n_features_to_select=20).fit(X, y))
report(f"RFE(Ridge), 20 of {p} kept, {n} x {p}", t_sk, t_us, float(np.abs(us.ranking_ - sk.ranking_).max()), "(ranking)")

## LassoLarsIC

In [ ]:
from sklearn.linear_model import LassoLarsIC
n, p = 100000, 100
X = rng.standard_normal((n, p)); y = X[:, :10] @ rng.standard_normal(10) + rng.standard_normal(n)
t_sk, sk = best(lambda: LassoLarsIC().fit(X, y), 1)
t_us, us = best(lambda: m.GramLassoLarsIC().fit(X, y))
report(f"LassoLarsIC (aic), {n} x {p}", t_sk, t_us, np.abs(us.coef_ - sk.coef_).max())

## On the GPU (GPU runtime only)

`DerivedLasso` runs the lasso the translator derived from a naive coordinate descent (`--derive --cublas`): a path of 20 penalties against scikit-learn's `lasso_path(precompute=True)`.

In [ ]:
from sklearn.linear_model import lasso_path
if m.cuda_available() and m.derived_backend() == "cublas":
    n, p = 100000, 200
    X = rng.standard_normal((n, p)); y = X[:, :10] @ rng.standard_normal(10) + rng.standard_normal(n)
    lams = m.DerivedLasso(X, y).lambda_grid(num=20, eps=1e-2)
    t_sk, (_, sk, _) = best(lambda: lasso_path(X, y, alphas=lams, precompute=True), 1)
    t_us, us = best(lambda: m.DerivedLasso(X, y).fit_path(lams))
    _, ref, _ = lasso_path(X, y, alphas=lams, precompute=True, tol=1e-12)
    report(f"lasso_path, 20 penalties, {n} x {p} (device)", t_sk, t_us, np.abs(us - ref.T).max(), "(vs a tight solve)")
else:
    print("no device (or no cuBLAS build): switch to a GPU runtime and run the notebook from the top")

## Summary

In [ ]:
print(f"{'':44} {'sklearn':>9} {'scm2cpp':>9} {'ratio':>7}")
for name, a, b, d, note in rows:
    print(f"{name:44} {a:8.3f}s {b:8.3f}s {a / b:6.1f}x")